# Basic Pinch and `dt_cont` Sensitivity
This notebook starts from the single-case `PinchProblem` front door, then uses `PinchWorkspace` for a named `dt_cont` sensitivity study on the packaged crude preheat train example.


## Case context
`crude_preheat_train.json` is a compact refinery preheat-train case with enough structure to show direct targets, graph interpretation, and how a small change in the minimum temperature approach assumption moves the utility picture. Notebook 4 picks up the real multistate derivative of this same case.


In [ ]:
import pandas as pd

from IPython.display import display

from OpenPinch import PinchProblem, PinchWorkspace


In [ ]:
problem = PinchProblem("crude_preheat_train.json", project_name="crude_preheat_train")
validated = problem.validate()
problem.target()
summary = problem.summary_frame()

{
    "stream_count": len(validated.streams),
    "utility_count": len(validated.utilities or []),
    "state_ids": list(problem.state_ids),
}


In [ ]:
summary


## Read the standard direct-integration graphs
The single-case API already exposes the graph catalog and the common figure accessors. For this process-only example, the direct-integration row is the main screen to interpret.


In [ ]:
catalog = problem.plot.catalog()
catalog.loc[
    catalog["Target"] == "crude_preheat_train/Direct Integration",
    ["Target", "Graph Type", "Graph Name"],
].reset_index(drop=True)


In [ ]:
display(problem.plot.composite_curve(zone_name="crude_preheat_train"))
display(problem.plot.shifted_composite_curve(zone_name="crude_preheat_train"))
display(problem.plot.grand_composite_curve(zone_name="crude_preheat_train"))


## Appendix: `area_cost()` return fields
The public target accessor also exposes the area-and-cost route. This cell reports the returned fields on the same real case so you can see what the current package build returns without leaving the main workflow.


In [ ]:
area_cost_target = problem.target.area_cost()
pd.DataFrame(
    [
        {
            "Target": area_cost_target.name,
            "Area": area_cost_target.area,
            "Number of Units": area_cost_target.num_units,
            "Capital Cost": area_cost_target.capital_cost,
            "Total Cost": area_cost_target.total_cost,
        }
    ]
)


## Named `dt_cont` sensitivity with `PinchWorkspace`
Once the single-case picture is clear, `PinchWorkspace` becomes the right tool for named baseline-versus-variant studies. Here the baseline stays intact while two copied cases tighten and widen the active `dt_cont` assumption.


In [ ]:
workspace = PinchWorkspace(
    source="crude_preheat_train.json",
    project_name="crude_preheat_train",
)

workspace.copy_case("baseline", "tight_dt", activate=False)
workspace.set_dt_cont_multiplier(0.5, case_name="tight_dt")
workspace.copy_case("baseline", "wide_dt", activate=False)
workspace.set_dt_cont_multiplier(2.0, case_name="wide_dt")

case_rows = []
for case_name, multiplier in [("tight_dt", 0.5), ("baseline", 1.0), ("wide_dt", 2.0)]:
    case_problem = workspace.case(case_name)
    case_problem.target()
    site_row = case_problem.summary_frame().loc[
        lambda frame: frame["Target"] == "crude_preheat_train/Direct Integration"
    ].iloc[0]
    case_rows.append(
        {
            "case": case_name,
            "dt_cont_multiplier": multiplier,
            "Hot Utility Target": site_row["Hot Utility Target"],
            "Cold Utility Target": site_row["Cold Utility Target"],
            "Heat Recovery": site_row["Heat Recovery"],
            "Hot Pinch": site_row["Hot Pinch"],
            "Cold Pinch": site_row["Cold Pinch"],
        }
    )

pd.DataFrame(case_rows).sort_values("dt_cont_multiplier").reset_index(drop=True)


In [ ]:
pd.concat(
    [
        workspace.compare_cases("baseline", case_name).loc[["Change"]].assign(case=case_name)
        for case_name in ["tight_dt", "wide_dt"]
    ],
    axis=0,
).reset_index(drop=True)


## Next step
This case is scalar, so the direct results above stay in one operating state. Notebook 4 switches to `crude_preheat_train_multistate.json` and shows the same direct targeting surface on real named states such as `turndown`, `base`, and `peak`.
